# 03. Análisis y Agrupaciones

### Dataset bancario limpio

**Objetivo de este notebook:** explorar el dataset ya limpio (`dataset_bancario_limpio.xlsx`) mediante agrupaciones (`groupby`) y tablas cruzadas (`crosstab`) para construir un **perfil del comportamiento financiero** de las empresas, bancos, categorías y niveles de riesgo.

No buscamos una sola pregunta puntual, sino ir armando evidencia: cada bloque de análisis va acompañado de:

1. Una celda de **instrucciones** (qué vamos a hacer y por qué).
2. El **código** que genera la tabla o el cruce.
3. Una celda de **insight / conclusión parcial** con la lectura de esos resultados.

Al final (sección 9) consolidamos todo en **conclusiones ejecutivas, alertas y recomendaciones**.

In [1]:
# 1. Carga del dataset limpio
import pandas as pd

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

# Cargar el dataset limpio
df = pd.read_excel("dataset_bancario_limpio.xlsx")

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (150, 33)


,transaccion_id,fecha_operacion,fecha_contable,empresa_id,empresa,cuenta_bancaria,banco,tipo_movimiento,categoria,proveedor_cliente,documento_ref,moneda,valor_bruto,impuesto_iva,valor_neto,metodo_pago,centro_costo,ciudad,pais,responsable,estado_conciliacion,fecha_vencimiento,dias_mora,nivel_riesgo,canal,observaciones,requiere_revision,motivo_revision,regla_signo_valida,regla_valor_neto_valida,regla_fechas_valida,regla_mora_valida,bandera_duplicado_conciliacion
0,TRX-001,2026-01-07,2026-01-07,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Capital,Egreso,Nomina,Servicios Beta,NOM-20260001,COP,-156933,0.00,-156933.00,Transferencia local,Comercial,Medellin,Colombia,Carlos Pena,Conciliado,2025-12-31,0,Bajo,PSE,NaN,False,No aplica,True,True,True,True,False
1,TRX-002,2026-01-10,2026-01-10,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Pacifico,Egreso,Impuestos,Aduanas Global,PAG-20260002,COP,-228866,0.00,-228866.00,Tarjeta credito,Operaciones,Cali,Colombia,Diana Torres,Conciliado,2026-01-08,0,Bajo,Transferencia,NaN,False,No aplica,True,True,True,True,False
2,TRX-003,2026-01-13,2026-01-13,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Nacional,Egreso,Servicios publicos,Energia Urbana,PAG-20260003,COP,-300799,-57151.81,-357950.81,Cheque,Tecnologia,Barranquilla,Colombia,Felipe Mora,Conciliado,2026-01-16,0,Bajo,Tarjeta,NaN,False,No aplica,True,True,True,True,False
3,TRX-004,2026-01-16,2026-01-17,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Andino,Egreso,Arriendo,Talento Humano,PAG-20260004,COP,-372732,-70819.08,-443551.08,Debito automatico,Finanzas,Cartagena,Colombia,Sin asignar,Pendiente,2026-01-24,159,Medio,Debito automatico,NaN,True,Conciliacion no cerrada,True,True,True,True,False
4,TRX-005,2026-01-19,2026-01-19,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Capital,Ingreso,Credito bancario,Arrendadora Centro,FAC-20260005,COP,444665,0.00,444665.00,ACH,Sin asignar,Bogota,Colombia,Ana Ruiz,Conciliado,2026-02-01,0,Bajo,Caja,NaN,False,No aplica,True,True,True,True,False


In [2]:
# Revisión rápida de columnas y tipos de dato
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   transaccion_id                  150 non-null    object        
 1   fecha_operacion                 150 non-null    datetime64[ns]
 2   fecha_contable                  150 non-null    datetime64[ns]
 3   empresa_id                      150 non-null    object        
 4   empresa                         150 non-null    object        
 5   cuenta_bancaria                 150 non-null    object        
 6   banco                           150 non-null    object        
 7   tipo_movimiento                 150 non-null    object        
 8   categoria                       150 non-null    object        
 9   proveedor_cliente               150 non-null    object        
 10  documento_ref                   150 non-null    object        
 11  moneda

**Nota:** confirmamos que `fecha_operacion`, `fecha_contable` y `fecha_vencimiento` quedaron como tipo fecha (`datetime`), y que las columnas `valor_bruto`, `valor_neto` y `dias_mora` son numéricas. Esto es indispensable para que las agregaciones (`sum`, `mean`, `max`) funcionen correctamente.

## 2. Análisis general

Antes de segmentar por empresa, banco o categoría, sacamos una fotografía general del dataset completo: cuántas transacciones hay, cuánto dinero se movió en total, cómo se reparten ingresos vs egresos, y cuál es el nivel de mora promedio de toda la operación. Esto nos da el punto de referencia contra el cual comparar cada segmento más adelante.

In [3]:
total_transacciones = len(df)
valor_total_bruto = df["valor_bruto"].sum()
valor_total_neto = df["valor_neto"].sum()
mora_promedio = df["dias_mora"].mean()

print(f"Total de transacciones: {total_transacciones}")
print(f"Valor total bruto:      {valor_total_bruto:,.2f}")
print(f"Valor total neto:       {valor_total_neto:,.2f}")
print(f"Mora promedio (días):   {mora_promedio:.2f}")

Total de transacciones: 150
Valor total bruto:      -154,140,259.00
Valor total neto:       -165,197,731.94
Mora promedio (días):   32.03


In [4]:
# Ingresos vs Egresos
resumen_tipo = df.groupby("tipo_movimiento").agg(
    cantidad=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    valor_promedio=("valor_neto", "mean")
)
resumen_tipo

,cantidad,valor_total,valor_promedio
tipo_movimiento,,,
Egreso,93,-4.233892e+08,-4.552572e+06
Ingreso,57,2.581915e+08,4.529675e+06


**Insight:** el dataset tiene **150 transacciones**, con un valor neto total de **-165,197,731.94**. Hay **93 egresos** frente a **57 ingresos**, es decir, el flujo de caja registrado en este período es predominantemente de salida de dinero. La mora promedio general es de **~32 días**, una cifra que por sí sola no dice mucho hasta que la comparemos por empresa, banco y nivel de riesgo en las siguientes secciones.


## 3. Análisis por empresa

Agrupamos por `empresa` para responder: ¿qué empresa concentra más transacciones?, ¿cuál mueve más dinero (en valor neto)?, ¿cuál tiene la mora promedio más alta? Esto sirve para detectar si el riesgo/mora está concentrado en una sola empresa o repartido de forma pareja.

In [5]:
analisis_empresa = df.groupby("empresa").agg(
    cantidad=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    valor_promedio=("valor_neto", "mean"),
    mora_promedio=("dias_mora", "mean"),
    mora_maxima=("dias_mora", "max")
).sort_values("valor_total")

analisis_empresa

,cantidad,valor_total,valor_promedio,mora_promedio,mora_maxima
empresa,,,,,
Logistica Prisma Ltda.,50,-59583015.10,-1.191660e+06,29.10,170
Insumos Horizonte S.A.,50,-55831110.42,-1.116622e+06,38.38,168
Nova Retail S.A.S.,50,-49783606.42,-9.956721e+05,28.62,136


In [10]:
# Separamos el volumen de dinero real (Ingresos vs Egresos) por empresa
# Así vemos el volumen real de dinero que entra y sale por empresa, no solo el neto.

flujo_por_empresa = (
    df.groupby(["empresa", "tipo_movimiento"])["valor_neto"]
    .agg(
        Transacciones="count",
        Monto_Total="sum",
        Promedio_Movimiento="mean",
    )
    .unstack(
        fill_value=0
    )  # Organiza ingresos y egresos en columnas lado a lado
)

print("--- Flujo de Caja Real por Empresa ---")
display(flujo_por_empresa)

--- Flujo de Caja Real por Empresa ---


Transacciones           Monto_Total              Promedio_Movimiento              
tipo_movimiento               Egreso Ingreso        Egreso      Ingreso              Egreso       Ingreso
empresa                                                                                                  
Insumos Horizonte S.A.            32      18 -1.375003e+08  81669181.38       -4.296884e+06  4.537177e+06
Logistica Prisma Ltda.            30      20 -1.451503e+08  85567254.71       -4.838342e+06  4.278363e+06
Nova Retail S.A.S.                31      19 -1.407386e+08  90955035.17       -4.539956e+06  4.787107e+06

**Insights:** 

- Las 3 empresas tienen exactamente **50 transacciones cada una** (muestra balanceada), pero no se comportan igual en mora: **Insumos Horizonte S.A.** tiene la mora promedio más alta (**38.4 días**) frente a **Logística Prisma Ltda.** (29.1) y **Nova Retail S.A.S.** (28.6). En valor movido, **Logística Prisma Ltda.** es la que más dinero mueve en términos absolutos (-59.6M), seguida de Insumos Horizonte (-55.8M) y Nova Retail (-49.8M). Esto sugiere que **Insumos Horizonte** merece seguimiento prioritario en gestión de cartera, no por volumen sino por lentitud de pago.
- Al separar ingresos y egresos por empresa se confirma que el valor neto negativo de cada compañía (visto en la sección 3) es producto de que **los egresos superan ampliamente a los ingresos** en las tres empresas — el detalle exacto de montos y transacciones queda visible en la tabla anterior, columna por columna (`Ingreso` / `Egreso`), en vez de quedar "escondido" en un solo neto compensado.

## 4. Análisis por banco

Agrupamos por `banco` para ver con cuál banco hay más operaciones, cuál concentra mayor valor transaccionado y cuál tiene la mora promedio más alta. Esto ayuda a identificar si algún banco en particular está asociado a más fricciones operativas.

In [11]:
analisis_banco = df.groupby("banco").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean")
).sort_values("valor_total", ascending=False)

analisis_banco

,transacciones,valor_total,mora_promedio
banco,,,
Banco Pacifico,38,-30176371.92,13.447368
Banco Nacional,37,-40693636.86,11.540541
Banco Capital,38,-44848882.92,13.131579
Banco Andino,37,-49478840.24,91.027027


**Insight:** los 4 bancos tienen un volumen de operaciones similar (37-38 transacciones cada uno), pero **Banco Andino** destaca claramente por una mora promedio de **~91 días**, muy por encima de Banco Pacífico (13.4), Banco Capital (13.1) y Banco Nacional (11.5). Esta diferencia es tan marcada que apunta a un problema puntual (posible cuenta, canal o tipo de operación específica) más que a una tendencia general del banco — vale la pena cruzarlo con `estado_conciliacion` y `nivel_riesgo` para aislar la causa.

## 5. Análisis por estado de conciliación

Esta es la vista más importante del dataset porque está orientado a control bancario. Agrupamos por `estado_conciliacion` para saber cuántas transacciones están conciliadas, cuánto dinero está pendiente, rechazado o en diferencia, y cuál es la mora asociada a cada estado.

In [12]:
analisis_conciliacion = df.groupby("estado_conciliacion").agg(
    cantidad=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean")
)
analisis_conciliacion

,cantidad,valor_total,mora_promedio
estado_conciliacion,,,
Conciliado,95,-91385485.57,0.000000
Diferencia,11,-11819394.12,79.181818
Duplicado,4,-17808115.96,77.000000
Pendiente,33,-27732137.64,95.151515
Rechazado,7,-16452598.65,69.428571


In [13]:
# Participación porcentual de cada estado sobre el total de transacciones
(df["estado_conciliacion"].value_counts(normalize=True) * 100).round(2)

estado_conciliacion
Conciliado    63.33
Pendiente     22.00
Diferencia     7.33
Rechazado      4.67
Duplicado      2.67
Name: proportion, dtype: float64

**Insight:** el **63.3%** de las transacciones están **Conciliado** (95 de 150, mora promedio 0 días, como es esperable). El resto —**36.7%**— está en algún estado de excepción: **Pendiente** (22.0%, con la mora promedio más alta de todas, ~95 días), **Diferencia** (7.3%, mora ~79 días), **Rechazado** (4.7%, mora ~69 días) y **Duplicado** (2.7%, mora ~77 días). El hallazgo clave es que **"Pendiente" concentra el mayor riesgo de mora**, no solo en cantidad de casos sino en severidad — es el segmento donde priorizar la gestión de conciliación.


## 6. Análisis por categoría

Agrupamos por `categoria` para ver en qué conceptos se concentra el dinero (nómina, impuestos, arriendo, ventas, crédito bancario, etc.), cuáles generan más egresos y cuáles podrían necesitar más revisión.

In [14]:
analisis_categoria = df.groupby("categoria").agg(
    cantidad=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum")
).sort_values("valor_total")

analisis_categoria

,cantidad,valor_total
categoria,,
Arriendo,11,-61488475.72
Pago proveedor,12,-53485235.44
Nomina,12,-52331452.00
Reembolso,12,-52138876.00
Servicios publicos,12,-51773230.04
Comision bancaria,12,-51275680.00
Impuestos,12,-48398360.00
Credito bancario,28,22809877.00
Venta internacional,19,85391458.00


In [15]:
# Cruce categoría x nivel de riesgo
pd.crosstab(df["categoria"], df["nivel_riesgo"])

nivel_riesgo,Alto,Bajo,Medio
categoria,,,
Arriendo,3,1,7
Comision bancaria,0,10,2
Credito bancario,0,21,7
Impuestos,0,9,3
Nomina,0,11,1
Pago proveedor,2,1,9
Reembolso,0,9,3
Servicios publicos,0,11,1
Venta internacional,1,12,6


**Insight:**

**Alta dependencia del crédito:** La principal fuente de liquidez del periodo fue el crédito bancario, por encima de las ventas nacionales e internacionales. Esto indica que las empresas están sosteniendo una parte importante de su operación mediante financiamiento directo del banco.

**Gasto operativo distribuido:** Los egresos recurrentes, como arriendo, pago a proveedores y nómina, se mantienen en niveles similares. No se evidencia un gasto único dominante, sino una carga operativa distribuida de forma relativamente homogénea.

**Déficit neto de operación:** Aunque existen entradas relevantes por créditos y ventas, el consolidado neto global cierra en negativo. Esto sugiere que una parte significativa del capital se está destinando a salidas no operativas, como pago de obligaciones previas, inversiones estructuradas o giros de tesorería. Esta dinámica justifica un monitoreo cercano por parte del banco ante un posible riesgo de liquidez.

## 7. Análisis de riesgo y mora

Agrupamos por `nivel_riesgo` para cuantificar cuántos casos hay en cada nivel, dónde se concentra la mora crítica (mora máxima) y cruzamos `estado_conciliacion` x `nivel_riesgo` para ver qué combinaciones son las más problemáticas.

In [17]:
analisis_riesgo = df.groupby("nivel_riesgo").agg(
    cantidad=("transaccion_id", "count"),
    mora_promedio=("dias_mora", "mean"),
    mora_maxima=("dias_mora", "max"),
    valor_total=("valor_neto", "sum")
)
analisis_riesgo

,cantidad,mora_promedio,mora_maxima,valor_total
nivel_riesgo,,,,
Alto,7,108.428571,168,-27605225.56
Bajo,101,0.930693,32,-98190996.85
Medio,42,94.095238,170,-39401509.53


In [18]:
# Cruce estado de conciliación x nivel de riesgo
pd.crosstab(df["estado_conciliacion"], df["nivel_riesgo"])

nivel_riesgo,Alto,Bajo,Medio
estado_conciliacion,,,
Conciliado,0,95,0
Diferencia,1,0,10
Duplicado,0,1,3
Pendiente,6,5,22
Rechazado,0,0,7


In [19]:
# ¿Requiere revisión manual? ¿cómo afecta la mora?
df.groupby("requiere_revision")["dias_mora"].agg(["count", "mean"])


,count,mean
requiere_revision,,
False,66,0.000000
True,84,57.202381


**Insight:** el riesgo **Bajo** concentra la mayoría de los casos (101 de 150) con mora prácticamente nula (0.93 días en promedio). Sin embargo, **Alto** (7 casos) y **Medio** (42 casos) tienen mora promedio de **108** y **94 días** respectivamente, con picos de hasta **170 días**. El cruce con conciliación confirma el patrón: **todo el riesgo Alto y Medio está en estados Pendiente, Diferencia o Rechazado** — ninguno está Conciliado. Además, las **84 transacciones que requieren revisión manual** tienen una mora promedio de **57.2 días**, contra **0 días** en las que no la requieren. Esto valida que el flag `requiere_revision` es un buen predictor temprano de mora.


## 8. Análisis temporal

Usamos `fecha_operacion` para agrupar por mes y ver cómo evolucionan la cantidad de transacciones, el valor movido y la mora promedio a lo largo del tiempo. Esto permite detectar si hay estacionalidad o si la mora está mejorando/empeorando mes a mes.

df["mes"] = df["fecha_operacion"].dt.to_period("M")

analisis_temporal = df.groupby("mes").agg(
    transacciones=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum"),
    mora_promedio=("dias_mora", "mean")
)
analisis_temporal


**Insight:** el volumen de transacciones se mantiene relativamente estable (19 a 31 por mes), pero la **mora promedio muestra una tendencia decreciente**: de 40-51 días en enero-febrero baja hasta **~5.6 días en junio**. Esto es una señal positiva — sugiere que la gestión de cobranza/conciliación mejoró a lo largo del semestre, o que las transacciones más recientes simplemente no han tenido tiempo de acumular mora todavía (algo a validar revisando la fecha de corte del dataset).


### 8.1 Cruces adicionales

Cerramos la exploración con cruces de dos variables que suelen revelar patrones que un solo `groupby` no muestra: empresa x estado de conciliación, y banco x estado de conciliación.

In [22]:
# Empresa x estado de conciliación
df.groupby(["empresa", "estado_conciliacion"]).size().unstack(fill_value=0)


estado_conciliacion,Conciliado,Diferencia,Duplicado,Pendiente,Rechazado
empresa,,,,,
Insumos Horizonte S.A.,30,7,0,10,3
Logistica Prisma Ltda.,33,4,1,10,2
Nova Retail S.A.S.,32,0,3,13,2


In [23]:
# Banco x estado de conciliación (cantidad y valor)
df.groupby(["banco", "estado_conciliacion"]).agg(
    cantidad=("transaccion_id", "count"),
    valor_total=("valor_neto", "sum")
)

cantidad  valor_total
banco          estado_conciliacion                       
Banco Andino   Diferencia                  2 -13555942.12
               Duplicado                   1  -1583989.96
               Pendiente                  33 -27732137.64
               Rechazado                   1  -6606770.52
Banco Capital  Conciliado                 32 -33408624.92
               Diferencia                  3  -4760645.00
               Duplicado                   1  -2746521.00
               Rechazado                   2  -3933092.00
Banco Nacional Conciliado                 31 -42527088.73
               Diferencia                  3   4711935.00
               Duplicado                   1  -8069563.00
               Rechazado                   2   5191079.87
Banco Pacifico Conciliado                 32 -15449771.92
               Diferencia                  3   1785258.00
               Duplicado                   1  -5408042.00
               Rechazado                   2 -11103816.00

**Insight:** estos cruces permiten ubicar exactamente **dónde** ocurren los estados problemáticos (Pendiente, Diferencia, Rechazado, Duplicado) desagregados por empresa y por banco — información clave para dirigir la gestión de conciliación a los puntos específicos de fricción en lugar de aplicar medidas genéricas a toda la cartera.

## 9. Conclusiones finales

### 5 hallazgos importantes

1. **El 36.7% de las transacciones no está conciliado** (Pendiente 22.0%, Diferencia 7.3%, Rechazado 4.7%, Duplicado 2.7%), y es justamente ese grupo el que concentra toda la mora del dataset.
2. **Banco Andino tiene una mora promedio de ~91 días**, muy por encima de los otros 3 bancos (11-13 días), pese a tener un volumen de operaciones similar.
3. **Insumos Horizonte S.A.** es la empresa con mayor mora promedio (~38.4 días), aunque no es la que más dinero mueve.
4. **Los niveles de riesgo Medio y Alto están 100% fuera del estado "Conciliado"** — ninguna transacción de riesgo Medio/Alto aparece como conciliada, lo que confirma que el flag de riesgo está bien calibrado.
5. **La mora promedio mensual bajó de ~51 días (febrero) a ~5.6 días (junio)**, sugiriendo una mejora progresiva en la gestión de conciliación a lo largo del semestre.

### 3 alertas

1. **Concentración de mora crítica en pocos casos:** solo 7 transacciones de riesgo Alto acumulan mora de hasta 170 días — bajo volumen, pero impacto severo si no se atienden pronto.
2. **Banco Andino como foco de fricción operativa:** su mora promedio (~91 días) es entre 7 y 8 veces mayor que la de los demás bancos, lo que amerita una revisión específica de esa relación bancaria.
3. **84 transacciones (56% del total) requieren revisión manual**, con una mora promedio de 57.2 días frente a 0 días en las que no la requieren — el proceso de revisión manual parece ser el cuello de botella principal.

### 3 recomendaciones

1. **Priorizar la gestión de la cartera "Pendiente"**, ya que es el estado con mayor mora promedio (~95 días) y el de mayor participación entre los no conciliados (22% del total).
2. **Auditar específicamente las operaciones con Banco Andino** para identificar la causa raíz de la mora elevada (¿un canal?, ¿un tipo de operación?, ¿una cuenta puntual?).
3. **Automatizar o agilizar el proceso de revisión manual** (`requiere_revision = True`), dado que es el factor más asociado a mora alta; reducir el tiempo de este paso debería traducirse directamente en menor mora promedio.